In [1]:
import pandas as pd 
import os 
import numpy as np 

In [2]:
chunks = [] 
for chunk in pd.read_csv("bill_schedule.csv", chunksize = 500_000):
    chunk['io_time'] = pd.to_datetime(chunk['io_time']) 
    chunks.append(chunk) 
sche = pd.concat(chunks, ignore_index = True) 
del chunk 
wh_1a = pd.read_csv('warehouse_1A.csv')
set_1a = set(wh_1a['name']) 

sche['next_wh'] = sche.groupby('bill_code')['warehouse_name'].shift(1) 
sche['pre_wh'] = sche.groupby('bill_code')['warehouse_name'].shift(-1) 
sche_first = sche[sche['warehouse_name'] != sche['next_wh']].copy() 
sche_last = sche[sche['warehouse_name'] != sche['pre_wh']].copy() 

sche_first['wh_rank'] = sche_first.groupby('bill_code').cumcount() 
sche_last['wh_rank'] = sche_last.groupby('bill_code').cumcount() 
sche_1a = sche_first[sche_first['warehouse_name'].isin(set_1a)] 
count_1a = sche_1a.groupby('bill_code').size().reset_index(name = 'count_1a') 
count_1a = count_1a[count_1a['count_1a'] >= 2] 
valid_bills = set(count_1a['bill_code']) 

sche_first = sche_first[sche_first['bill_code'].isin(valid_bills)] 
max_rank_first = sche_first.groupby('bill_code')['wh_rank'].transform('max') 
sche_first['wh_rank_rev'] = max_rank_first - sche_first['wh_rank'] 

sche_last = sche_last[sche_last['bill_code'].isin(valid_bills)] 
max_rank_last = sche_last.groupby('bill_code')['wh_rank'].transform('max') 
sche_last['wh_rank_rev'] = max_rank_last - sche_last['wh_rank'] 

sche_first_1a = sche_first[(sche_first['wh_rank'] > 0) & (sche_first['warehouse_name'].isin(set_1a))] 
wh_o1a_first = sche_first_1a.drop_duplicates(subset=['bill_code'], keep='first') 
wh_o1a = (wh_o1a_first[['bill_code', 'warehouse_name', 'io_time', 'io_status']]
          .rename(columns={'warehouse_name': 'kho_o1a', 'io_time': 'time_o1a', 'io_status': 'status_o1a'})) 

In [3]:
data = wh_o1a['kho_o1a'].value_counts() 
import plotly.express as px
fig = px.bar(data) 
fig.show()